<a href="https://colab.research.google.com/github/zakateryila/Zaka-Teryila-Peter-/blob/main/Day_6_Lab_Fetch%2C_Parse%2C_Transform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json

# Load the available sample JSON file
with open('/content/sample_data/anscombe.json', 'r') as f:
    data = json.load(f)

print(json.dumps(data, indent=2)[:1000])

[
  {
    "Series": "I",
    "X": 10.0,
    "Y": 8.04
  },
  {
    "Series": "I",
    "X": 8.0,
    "Y": 6.95
  },
  {
    "Series": "I",
    "X": 13.0,
    "Y": 7.58
  },
  {
    "Series": "I",
    "X": 9.0,
    "Y": 8.81
  },
  {
    "Series": "I",
    "X": 11.0,
    "Y": 8.33
  },
  {
    "Series": "I",
    "X": 14.0,
    "Y": 9.96
  },
  {
    "Series": "I",
    "X": 6.0,
    "Y": 7.24
  },
  {
    "Series": "I",
    "X": 4.0,
    "Y": 4.26
  },
  {
    "Series": "I",
    "X": 12.0,
    "Y": 10.84
  },
  {
    "Series": "I",
    "X": 7.0,
    "Y": 4.81
  },
  {
    "Series": "I",
    "X": 5.0,
    "Y": 5.68
  },
  {
    "Series": "II",
    "X": 10.0,
    "Y": 9.14
  },
  {
    "Series": "II",
    "X": 8.0,
    "Y": 8.14
  },
  {
    "Series": "II",
    "X": 13.0,
    "Y": 8.74
  },
  {
    "Series": "II",
    "X": 9.0,
    "Y": 8.77
  },
  {
    "Series": "II",
    "X": 11.0,
    "Y": 9.26
  },
  {
    "Series": "II",
    "X": 14.0,
    "Y": 8.1
  },
  {
    "Series": "II",
    "X"

In [2]:
import requests
BASE_URL = "https://api.frankfurter.dev/v1/latest"

def fetch_rates(base="USD"):
    params = {"base": base}
    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        response.raise_for_status()
    except requests.exceptions.Timeout:
        print("The request timed out.")
        return None
    except requests.exceptions.HTTPError as err:
        print(f"The API rejected the request: {err}")
        return None
    except requests.exceptions.RequestException as err:
        print(f"Could not reach the service: {err}")
        return None
    return response.json()

In [3]:
def to_records(data):
    rates = data.get("rates")
    if not rates:
        print("No 'rates' in the response.")
        return []
    return [{"currency": code, "rate": value} for code, value in rates.items()]

In [4]:
# Fetch latest currency rates relative to USD
raw_data = fetch_rates(base="USD")

if raw_data:
    # Transform the dict of rates into a flat list of records
    records = to_records(raw_data)

    if records:
        # Sort the records by exchange rate value relative to 1 USD
        # Higher rate value means you get more of that currency per USD, indicating a weaker currency.
        # Lower rate value means it is stronger (closer to, or worth more than, 1 USD).
        sorted_records = sorted(records, key=lambda x: x['rate'])

        strongest = sorted_records[0]
        weakest = sorted_records[-1]
        spread = weakest['rate'] - strongest['rate']

        print(f"--- Currency Analysis relative to Base (USD) ---\n")
        print(f"Strongest Currency: {strongest['currency']} at {strongest['rate']:.4f} per USD")
        print(f"Weakest Currency:   {weakest['currency']} at {weakest['rate']:.4f} per USD")
        print(f"Exchange Rate Spread: {spread:.4f} between {weakest['currency']} and {strongest['currency']}")
        print("\nFull sorted rates (Strongest to Weakest relative to USD):")
        for item in sorted_records:
            print(f"  {item['currency']}: {item['rate']:.4f}")
    else:
        print("Could not process records.")
else:
    print("Could not fetch data from the API.")

--- Currency Analysis relative to Base (USD) ---

Strongest Currency: GBP at 0.7572 per USD
Weakest Currency:   IDR at 17920.0000 per USD
Exchange Rate Spread: 17919.2428 between IDR and GBP

Full sorted rates (Strongest to Weakest relative to USD):
  GBP: 0.7572
  CHF: 0.8337
  EUR: 0.8940
  SGD: 1.2820
  CAD: 1.4262
  AUD: 1.4402
  NZD: 1.7892
  ILS: 3.0773
  PLN: 3.9114
  MYR: 4.0915
  RON: 4.7769
  BRL: 5.0168
  DKK: 6.6815
  CNY: 6.7023
  HKD: 7.8476
  NOK: 9.5807
  SEK: 10.0072
  ZAR: 16.6505
  MXN: 18.0127
  CZK: 21.8160
  THB: 33.6850
  TRY: 49.2150
  PHP: 63.0030
  INR: 96.7800
  ISK: 122.4700
  JPY: 158.2800
  HUF: 327.4200
  KRW: 1343.4600
  IDR: 17920.0000


In [5]:
import csv
def write_csv(records, path):
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0].keys()))
        writer.writeheader()
        writer.writerows(records)
    print(f"Wrote {len(records)} rows to {path}")

### Step 6: Break it on purpose (Graceful Degradation Tests)

In [6]:
import requests

# 1. Test Misspelled Host
print("--- Test 1: Misspelled Host ---")
temp_broken_url = "https://api.frankfurter.invalid/v1/latest"
try:
    response = requests.get(temp_broken_url, params={"base": "USD"}, timeout=10)
    response.raise_for_status()
except requests.exceptions.Timeout:
    print("The request timed out.")
except requests.exceptions.HTTPError as err:
    print(f"The API rejected the request: {err}")
except requests.exceptions.RequestException as err:
    print(f"Could not reach the service: {err}")

# 2. Test Immediate Timeout (timeout=0.001)
print("\n--- Test 2: Timeout (timeout=0.001) ---")
try:
    response = requests.get(BASE_URL, params={"base": "USD"}, timeout=0.001)
    response.raise_for_status()
except requests.exceptions.Timeout:
    print("The request timed out.")
except requests.exceptions.HTTPError as err:
    print(f"The API rejected the request: {err}")
except requests.exceptions.RequestException as err:
    print(f"Could not reach the service: {err}")

# 3. Test Invalid Base Currency (deciding to handle and print API error gracefully)
print("\n--- Test 3: Invalid Base Currency ---")
try:
    response = requests.get(BASE_URL, params={"base": "INVALID_CURRENCY"}, timeout=10)
    response.raise_for_status()
except requests.exceptions.Timeout:
    print("The request timed out.")
except requests.exceptions.HTTPError as err:
    print(f"The API rejected the request: {err}")
except requests.exceptions.RequestException as err:
    print(f"Could not reach the service: {err}")

--- Test 1: Misspelled Host ---
Could not reach the service: HTTPSConnectionPool(host='api.frankfurter.invalid', port=443): Max retries exceeded with url: /v1/latest?base=USD (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7ab65bd0cf50>: Failed to resolve 'api.frankfurter.invalid' ([Errno -2] Name or service not known)"))

--- Test 2: Timeout (timeout=0.001) ---
Could not reach the service: HTTPSConnectionPool(host='api.frankfurter.dev', port=443): Max retries exceeded with url: /v1/latest?base=USD (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7ab65bd0df90>: Failed to establish a new connection: [Errno 101] Network is unreachable'))

--- Test 3: Invalid Base Currency ---
The API rejected the request: 404 Client Error: Not Found for url: https://api.frankfurter.dev/v1/latest?base=INVALID_CURRENCY
